# RoBERTa language-metrics correlations

Transformer counterpart of the paper's **Table `tab:js-correlation`** (Language Metrics section),
which correlates — over the 12 off-diagonal dataset pairs — the **JS-divergence between datasets'
training distributions** (Words, POS, entity span length, entity span-based counts) against the
**cross-dataset span-F1 performance matrix**.

Only the performance matrix changes between BiLSTM and RoBERTa. The JS-divergence matrices are
**model- and dedup-independent**: the paper used the `Train-to-Train` variant, computed purely from
the datasets' training text and never touching leakage removal. So we reuse those matrices verbatim
from `languagemetrics.ipynb` (`all_results`, also in `langaugemetricssheets/divergence_results.xlsx`)
and only swap the performance matrix.

The BiLSTM column is kept as a sanity check: recomputing it must reproduce the paper's
-0.74 / -0.71 / -0.36 / 0.00.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

## Cached Train-to-Train JS matrices

Reproduced verbatim from `languagemetrics.ipynb`. Dataset order (rows/cols) is
**DNRTI, Attacker, APTNER, CyNER** — the same order used by the performance matrices below.

In [ ]:
# JS-divergence between training-set distributions (paper 'Train-to-Train' variant).
# Source: languagemetrics.ipynb -> all_results[<dist>]['Train-to-Train_JS'].
js = {
    "Words": np.array([
        [0.        , 0.35760725, 0.13831401, 0.40274952],
        [0.35760725, 0.        , 0.3357986 , 0.37419193],
        [0.13831401, 0.3357986 , 0.        , 0.36833671],
        [0.40274952, 0.37419193, 0.36833671, 0.        ],
    ]),
    "POS labels": np.array([
        [0.        , 0.04199404, 0.02111907, 0.05641533],
        [0.04199404, 0.        , 0.04111081, 0.0438426 ],
        [0.02111907, 0.04111081, 0.        , 0.04155997],
        [0.05641533, 0.0438426 , 0.04155997, 0.        ],
    ]),
    "Entity span lengths": np.array([
        [0.        , 0.22936515, 0.03539619, 0.05009848],
        [0.22936515, 0.        , 0.24052952, 0.19019921],
        [0.03539619, 0.24052952, 0.        , 0.06866643],
        [0.05009848, 0.19019921, 0.06866643, 0.        ],
    ]),
    "Entity span-based counts": np.array([
        [0.        , 0.13617656, 0.31196765, 0.3691458 ],
        [0.13617656, 0.        , 0.28220034, 0.27089175],
        [0.31196765, 0.28220034, 0.        , 0.42346006],
        [0.3691458 , 0.27089175, 0.42346006, 0.        ],
    ]),
}

## Performance matrices

Rows = train dataset, cols = eval/dev dataset; order = DNRTI, Attacker, APTNER, CyNER.
BiLSTM from paper Table `tab:cross_eval_matrix`; RoBERTa is the cross-dataset span-F1 from
`models/cross_retrained/REPORT.md` ("RoBERTa - slot-f1, valid, prob-sum late merge, leakage-clean"),
also in `reports/cross_dataset_roberta_results.md`.

In [ ]:
perf = {
    "BiLSTM": np.array([
        [0.41, 0.16, 0.19, 0.07],
        [0.09, 0.23, 0.01, 0.02],
        [0.31, 0.16, 0.41, 0.18],
        [0.05, 0.04, 0.06, 0.40],
    ]),
    "RoBERTa": np.array([
        [0.63, 0.30, 0.21, 0.16],
        [0.34, 0.61, 0.26, 0.37],
        [0.39, 0.32, 0.59, 0.55],
        [0.25, 0.24, 0.32, 0.73],
    ]),
}

## Correlations

Pearson correlation over the 12 **off-diagonal** cells (cross-dataset pairs only), exactly as in the
paper. The BiLSTM column must reproduce the paper's -0.74 / -0.71 / -0.36 / 0.00.

In [ ]:
MASK = ~np.eye(4, dtype=bool)  # off-diagonal only

def corr(perf_matrix, js_matrix):
    r, p = pearsonr(perf_matrix[MASK], js_matrix[MASK])
    return r, p

ORDER = ["Words", "POS labels", "Entity span lengths", "Entity span-based counts"]
PAPER_BILSTM = {"Words": -0.74, "POS labels": -0.71,
                "Entity span lengths": -0.36, "Entity span-based counts": 0.00}

rows = []
for dist in ORDER:
    r_b, _   = corr(perf["BiLSTM"],  js[dist])
    r_r, p_r = corr(perf["RoBERTa"], js[dist])
    rows.append({
        "Distribution": dist,
        "BiLSTM (paper)": PAPER_BILSTM[dist],
        "BiLSTM (recomputed)": round(r_b, 2),
        "RoBERTa": round(r_r, 2),
        "RoBERTa p-value": round(p_r, 2),
    })

table = pd.DataFrame(rows)
assert np.allclose(table["BiLSTM (recomputed)"], [PAPER_BILSTM[d] for d in ORDER], atol=0.01), \
    "BiLSTM correlations do not reproduce the paper -- pipeline mismatch"
print("Pipeline validated: BiLSTM correlations reproduce paper Table tab:js-correlation.\n")
table

## Result & interpretation

The strong negative correlations reported for the BiLSTM **collapse to near zero for RoBERTa**
(and none are statistically significant). Where a from-scratch BiLSTM transferred worse between
datasets whose word/POS distributions diverged more (Pearson -0.74 / -0.71), the pretrained RoBERTa
encoder is largely **robust to that distributional shift**, so JS-divergence no longer predicts
cross-dataset performance.

**Caveat.** Only the performance matrix differs; the JS matrices are identical. The RoBERTa perf
matrix uses union-leakage removal and prob-sum late-merge to the coarse label set, whereas the
BiLSTM was unified-trained — a confound to note when writing this up. Unlike the other RoBERTa
tables, this one is **not a clean drop-in**: the paper's accompanying prose (word/POS divergence
"correlated negatively with performance") reverses and needs rewriting.